# HazeWaveNet - Kaggle Multi-GPU Training

Notebook n?y d?nh cho Kaggle v?i 2 GPU. D? li?u ???c ??c t? `/kaggle/input`, k?t qu?/checkpoint ghi v?o `/kaggle/working`. Cell setup t? d? th? m?c `clear/hazy` v? c?u tr?c dataset tr?n Kaggle c? th? c? th?m m?t l?p th? m?c l?ng nhau.

In [ ]:
# Cell 1 - Install dependencies and locate the repository
!pip -q install pillow matplotlib numpy tqdm thop openpyxl

import os
import sys
import subprocess
from pathlib import Path

KAGGLE_INPUT = Path("/kaggle/input")
WORKDIR = Path("/kaggle/working/VDT_GD2")
if not WORKDIR.exists():
    # Clone the project when Kaggle only receives this notebook.
    subprocess.run(["git", "clone", "https://github.com/Darkin72/VDT_GD2.git", str(WORKDIR)], check=True)
os.chdir(WORKDIR)
if str(WORKDIR) not in sys.path:
    sys.path.insert(0, str(WORKDIR))
print("Repository:", WORKDIR)


In [ ]:
# Cell 2 - Find ITS/OTS roots in Kaggle's mounted dataset layout
from pathlib import Path

def image_count(folder):
    return sum(p.is_file() for p in folder.iterdir()) if folder.is_dir() else 0

def find_paired_root(dataset_name, prefer_indoor=False):
    candidates = []
    for clear_dir in KAGGLE_INPUT.rglob("clear"):
        hazy_dir = clear_dir.parent / "hazy"
        if not hazy_dir.is_dir():
            continue
        text = str(clear_dir).lower()
        score = image_count(clear_dir) + image_count(hazy_dir)
        if dataset_name in text:
            score += 100000
        if prefer_indoor and "indoor" in text:
            score += 10000
        candidates.append((score, clear_dir.parent))
    if not candidates:
        raise FileNotFoundError(f"Kh?ng t?m th?y c?p clear/hazy c?a {dataset_name} trong {KAGGLE_INPUT}")
    return max(candidates, key=lambda item: item[0])[1]

ITS_ROOT = find_paired_root("its", prefer_indoor=True)
OTS_ROOT = find_paired_root("ots")
print("ITS:", ITS_ROOT, "clear=", image_count(ITS_ROOT / "clear"), "hazy=", image_count(ITS_ROOT / "hazy"))
print("OTS:", OTS_ROOT, "clear=", image_count(OTS_ROOT / "clear"), "hazy=", image_count(OTS_ROOT / "hazy"))


In [ ]:
# Cell 3 - Multi-GPU training
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required")
GPU_COUNT = torch.cuda.device_count()
if GPU_COUNT < 2:
    raise RuntimeError(f"Expected 2 GPUs, found {GPU_COUNT}")
print("GPUs:", GPU_COUNT, [torch.cuda.get_device_name(i) for i in range(GPU_COUNT)])

COMMON = [
    sys.executable, "-m", "solution.wavelet_dehaze.train",
    "--device", "cuda", "--multi-gpu", "--epochs", "1000",
    "--batch-size", "16", "--num-workers", "2", "--size", "256", "--weight-decay", "1e-4",
    "--scheduler", "cosine", "--min-lr", "1e-6", "--augment", "--multi-scale",
    "--patience", "50",
]

def run_training(data_root, output_name, history_name, lr):
    command = COMMON + [str(data_root), "--lr", str(lr), "--out", output_name, "--history", history_name]
    print("\nRunning:", " ".join(command))
    subprocess.run(command, cwd=WORKDIR, check=True)

run_training(ITS_ROOT, "/kaggle/working/haze_wavelet_sots_its.pt", "/kaggle/working/history_sots_its.json", 2e-4)
run_training(OTS_ROOT, "/kaggle/working/haze_wavelet_sots_ots.pt", "/kaggle/working/history_sots_ots.json", 1e-4)


In [ ]:
# Cell 4 - Training curves
import json
import matplotlib.pyplot as plt

histories = {}
for label, path in {"SOTS-ITS": "/kaggle/working/history_sots_its.json", "SOTS-OTS": "/kaggle/working/history_sots_ots.json"}.items():
    path = Path(path)
    if path.exists():
        histories[label] = json.loads(path.read_text())
fig, axes = plt.subplots(len(histories), 3, figsize=(18, 5 * max(1, len(histories))), squeeze=False)
for row, (label, records) in enumerate(histories.items()):
    epochs = [r["epoch"] for r in records]
    for col, (train_key, val_key, title) in enumerate((("train_loss", "val_loss", "Loss"), ("train_psnr", "val_psnr", "PSNR"), ("train_ssim", "val_ssim", "SSIM"))):
        ax = axes[row, col]
        ax.plot(epochs, [r[train_key] for r in records], label="Train")
        if val_key in records[0]: ax.plot(epochs, [r[val_key] for r in records], label="Validation")
        ax.set_title(f"{label} - {title}"); ax.set_xlabel("Epoch"); ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.savefig("/kaggle/working/haze_wavelet_kaggle_metrics.png", dpi=160); plt.show()


In [ ]:
# Cell 5 - Params/FLOPs hardware report
import pandas as pd
from thop import profile
from solution.wavelet_dehaze.model import HazeWaveNet

rows = []
for side in (256, 1024):
    model = HazeWaveNet().cuda().eval()
    checkpoint = "/kaggle/working/haze_wavelet_sots_its.pt"
    state = torch.load(checkpoint, map_location="cuda")
    model.load_state_dict(state.get("model", state))
    params = sum(p.numel() for p in model.parameters())
    macs, _ = profile(model, inputs=(torch.zeros(1, 3, side, side, device="cuda"),), verbose=False)
    rows.append({"model": "HazeWaveNet", "input": f"1x3x{side}x{side}", "parameters": params, "gmacs": macs / 1e9, "gflops": 2 * macs / 1e9, "gpu_count": GPU_COUNT, "gpu": torch.cuda.get_device_name(0)})
hardware = pd.DataFrame(rows)
hardware.to_csv("/kaggle/working/haze_wavelet_hardware.csv", index=False)
hardware.to_excel("/kaggle/working/haze_wavelet_hardware.xlsx", index=False)
hardware
